In [ ]:
# notebook cells, roughly:
# 1. load df_rg, region_by_id, df_esm
# 2. X_static, y, groups, factory = assemble_everything(...)   # from assemble_features
# 3. full = run_nested_cv(...)                                  # Goal 1: headline AUC
# 4. results = run_group_configs(..., logo + isolation configs) # Goal 2
#    plot_group_rocs(results, y)
# 5. permutation / SHAP                                          # importance
# 6. fit_final_model(...); apply_to_new_set(...)                # Goal 3

In [ ]:
%load_ext autoreload
%autoreload 2

import pandas as pd, numpy as np
import json
from src.classifier.assemble_features import assemble_everything
from src.classifier.feature_groups import FEATURE_GROUPS, SUPERGROUPS
from src.classifier.nested_cv import run_nested_cv, report_ungrouped

from src.analysis_visualization.plot_config import (
    GROUP_COLORS, save_figure, significance_stars, FIGSIZE_SINGLE
)

RATES_PATH = "/mnt/d/phd/scripts/16_ev_signature_predictor/data/samocha_mutation_rates/fordist_1KG_mutation_rate_table.txt"

# df_rg, region_by_id, df_esm already loaded from your pipeline

In [ ]:
DATA_DIR = "/mnt/d/phd/scripts/16_ev_signature_predictor/data/processed"

df_for_rg = pd.read_parquet(f"{DATA_DIR}/variants_annotated_final.parquet")
print(f"Loaded {len(df_for_rg):,} variant-region assignments")
print(f"Columns ({len(df_for_rg.columns)}): {df_for_rg.columns.tolist()}")


# Also load the regions JSON — useful for context (WT sequences, etc.)
with open(f"{DATA_DIR}/genomic_coords_merged_win5.json") as f:
    regions = json.load(f)
region_by_id = {r["region_id"]: r for r in regions}
print(f"Loaded {len(regions)} regions from JSON")


# df_esm = pd.read_parquet(f"{DATA_DIR}/variants_with_esm.parquet")

In [ ]:
X_static, y, groups, factory = assemble_everything(
    df_for_rg, region_by_id, # df_esm=df_esm,
    rates_path=RATES_PATH,
    codon_source_aas=list("ADEGLPRS"),
    codon_rates=None,
    alpha=0.5, split_source=True,
    physchem_delta_cache="/mnt/d/phd/scripts/16_ev_signature_predictor/data/processed/physchem_deltas.parquet",
    sub_from_col="before_aa",      # <-- match compute_substitution_counts
    sub_to_col="after_aa",         # <-- match compute_substitution_counts
)

print("X_static:", X_static.shape, "| regions:", len(X_static), "| accessions:", groups.nunique())
print("label balance:", y.value_counts().to_dict())
print("\nungrouped columns (in matrix, in NO group — would never be tested):")
print(report_ungrouped(X_static.columns))

In [ ]:
# full = run_nested_cv(
#     X_static, y, groups,
#     include_groups=list(FEATURE_GROUPS),
#     folded_transformer_factory=factory,
#     n_splits=20, n_trees=300,
# )
# print(f"FULL model: AUC {full['mean_auc']:.3f} ± {full['std_auc']:.3f}")
# print("per-fold:", [round(a,3) for a in full['per_fold_auc']])
# print("static cols used:", full['n_static_cols'], "| folded used:", full['used_folded'])

In [ ]:
static_groups = [g for g in FEATURE_GROUPS if g != "substitution_score"]
static_only = run_nested_cv(
    X_static, y, groups,
    include_groups=static_groups,
    folded_transformer_factory=None,   # no folded feature needed
    n_splits=5, n_trees=300,
)
print(f"Static-only: AUC {static_only['mean_auc']:.3f} ± {static_only['std_auc']:.3f}")
print("per-fold:", [round(a,3) for a in static_only['per_fold_auc']])

In [ ]:
from src.classifier.group_analysis import run_group_comparison, plot_group_rocs, plot_necessity_sufficiency

results, table = run_group_comparison(
    X_static, y, groups, factory=factory,
    n_splits=5, n_trees=300,        # full group set (omit group_names)
)
table.to_csv("/mnt/d/phd/scripts/16_ev_signature_predictor/data/output/group_RF_comparison.csv", index=False)

# # ROC: full + each group alone (or pick a subset to keep it readable)
# iso_labels = ["FULL (all groups)"] + [f"{g} only" for g in FEATURE_GROUPS]
# fig = plot_group_rocs(results, y, configs_to_show=iso_labels)
# # save_figure(fig, "group_roc")

# # fig.savefig("/mnt/d/phd/scripts/16_ev_signature_predictor/figures/group_roc.png", dpi=200, bbox_inches="tight")
# # fig.savefig("/mnt/d/phd/scripts/16_ev_signature_predictor/figures/group_roc.svg", dpi=200, bbox_inches="tight")

# fig2 = plot_necessity_sufficiency(table)
# save_figure(fig2, "group_necessity_sufficiency")

In [ ]:
import pickle
path_prefix = "/mnt/d/phd/scripts/16_ev_signature_predictor/data/output/group_RF_comparison"

table.to_parquet(f"{path_prefix}_table.parquet", index=False)
with open(f"{path_prefix}_results.pkl", "wb") as fh:
    pickle.dump(results, fh)
print(f"saved {path_prefix}_table.parquet and {path_prefix}_results.pkl")

In [ ]:
path_prefix = "/mnt/d/phd/scripts/16_ev_signature_predictor/data/output/group_RF_comparison"

table = pd.read_parquet(f"{path_prefix}_table.parquet")
with open(f"{path_prefix}_results.pkl", "rb") as fh:
    results = pickle.load(fh)

In [ ]:
from src.classifier.importance import (
    group_permutation_importance, fit_full_model_for_shap, shap_importance,
    plot_group_importance, plot_shap_summary, plot_shap_bar)

# --- group permutation importance (the headline) ---
gpi_df = group_permutation_importance(
    X_static, y, groups,
    include_groups=[g for g in FEATURE_GROUPS if g != "substitution_score"],  # fold-static
    n_splits=5, n_trees=300, n_repeats=10)
print(gpi_df.round(4).to_string(index=False))
# fig = plot_group_importance(gpi_df)
# # fig.savefig("/mnt/d/phd/scripts/16_ev_signature_predictor/figures/group_permutation_importance.png", dpi=200, bbox_inches="tight")
# save_figure(fig, "group_permutation_importance")


In [ ]:
# from src.classifier.importance import plot_shap_bar_by_group, plot_shap_beeswarm_grouped


from src.classifier.importance import get_group_colors, plot_shap_beeswarm_grouped
from src.classifier.feature_groups import FEATURE_GROUPS

rf, Xi_df, names, imp = fit_full_model_for_shap(
    X_static, y,
    include_groups=[g for g in FEATURE_GROUPS if g != "substitution_score"])

sv, shap_summary = shap_importance(rf, Xi_df)

# # # grouped bar (clean, recommended for the paper)
# # fig, gcolors = plot_shap_bar_by_group(shap_summary, max_display=20)
# # save_figure(fig, "shap_by_group")
# # fig.savefig("shap_by_group.png", dpi=200, bbox_inches="tight")

# # grouped beeswarm (richer, shows direction) — reuse the SAME colors for consistency
# fig2, _ = plot_shap_beeswarm_grouped(sv, Xi_df, max_display=20, group_colors=gcolors)
# save_figure(fig2, "shap_beeswarm_by_group")
# # fig2.savefig("shap_beeswarm_by_group.png", dpi=200, bbox_inches="tight")






# fig2, _ = plot_shap_beeswarm_grouped(sv, Xi_df, max_display=20, group_colors=group_colors)
# save_figure(fig2, "shap_beeswarm_by_group")

In [ ]:
group_colors = get_group_colors(list(FEATURE_GROUPS.keys()))

iso_labels = ["FULL (all groups)"] + [f"{g} only" for g in FEATURE_GROUPS]

fig  = plot_group_rocs(results, y, configs_to_show=iso_labels, group_colors=group_colors)
save_figure(fig, "group_roc")
fig2 = plot_necessity_sufficiency(table, group_colors=group_colors)
save_figure(fig2, "group_necessity_sufficiency")
fig3 = plot_group_importance(gpi_df, group_colors=group_colors)
save_figure(fig3, "group_importance")
fig4, _ = plot_shap_beeswarm_grouped(sv, Xi_df, max_display=20, group_colors=group_colors)
save_figure(fig4, "group_shap")

In [ ]:
from src.classifier.final_model import train_final_model, save_model

bundle = train_final_model(X_static, y, n_trees=300)
save_model(bundle, "/mnt/d/phd/scripts/16_ev_signature_predictor/models/rg_classifier_final.joblib")

In [ ]:
import numpy as np
import pandas as pd
import shap
from sklearn.inspection import permutation_importance
from sklearn.metrics import roc_auc_score, make_scorer

from src.classifier.feature_groups import FEATURE_GROUPS
from src.classifier.final_model import load_model

# ── load the deployable bundle ──────────────────────────────────────────────
bundle = load_model("/mnt/d/phd/scripts/16_ev_signature_predictor/models/rg_classifier_final.joblib")
rf        = bundle["rf"]
imp       = bundle["imputer"]
feat_cols = bundle["feature_cols"]      # 111 retained, in model order
pruned    = set(bundle["pruned_cols"])  # 16 dropped by correlation filter

# all candidate features = retained + pruned (the full pre-prune set)
all_feats = list(feat_cols) + [c for c in pruned if c not in feat_cols]

# ── 1) feature -> group map (from feature_groups.py) ────────────────────────
feat_to_group = {}
for grp, cols in FEATURE_GROUPS.items():
    for c in cols:
        feat_to_group[c] = grp   # if a feature is listed in two groups, last wins

# ── 2) hand-written descriptions (EDIT THESE) ───────────────────────────────
# Scaffolded by group; fill/refine wording. Any feature missing here gets "".
DESCRIPTIONS = {
    "region_length": "Length of the motif region in residues.",
    "n_rg_motifs": "Number of RG/RGG di-residue motifs in the region.",
    "rg_fraction": "Fraction of the region occupied by RG/RGG motifs.",
    "wt_ncpr": "Wild-type net charge per residue.",
    "wt_fcr": "Wild-type fraction of charged residues.",
    "wt_kappa": "Wild-type charge-patterning (kappa).",
    "wt_hydropathy": "Wild-type mean hydropathy.",
    "wt_aromaticity": "Wild-type aromatic-residue fraction.",
    "wt_fraction_proline": "Wild-type proline fraction.",
    "wt_n_pos": "Wild-type count of positively charged residues.",
    "wt_n_neg": "Wild-type count of negatively charged residues.",
    "gc": "Region GC content.",
    "gc3": "GC content at codon third positions (wobble).",
    "cpg_frac": "Fraction of dinucleotides that are CpG.",
    "cpg_oe": "Observed/expected CpG ratio.",
    "codon_mean_mutability": "Mean intrinsic codon mutability (trinucleotide rates).",
    "density_synonymous": "Synonymous variants per residue.",
    "density_missense": "Missense variants per residue.",
    "fraction_missense": "Missense fraction of all variants.",
    "density_inframe_indel": "In-frame indels per residue.",
    "fraction_inframe_indel": "In-frame indel fraction of all variants.",
    "density_LoF": "Loss-of-function variants per residue.",
    "fraction_LoF": "LoF fraction of all variants.",
    "n_LoF": "Count of LoF variants.",
    "n_other": "Count of other-consequence variants.",
    "n_variants_total": "Total variants in the region.",
    "variant_density": "Total variants per residue.",
    "am_median": "Median AlphaMissense pathogenicity (missense).",
    "am_mean": "Mean AlphaMissense pathogenicity.",
    "am_max": "Maximum AlphaMissense pathogenicity.",
    "am_std": "SD of AlphaMissense pathogenicity.",
    "am_fraction_pathogenic": "Fraction of missense classed pathogenic by AlphaMissense.",
    "esm_mean": "Mean ESM1b LLR (missense).",
    "esm_median": "Median ESM1b LLR.",
    "esm_min": "Minimum (most disruptive) ESM1b LLR.",
    "esm_std": "SD of ESM1b LLR.",
    "esm_fraction_disruptive": "Fraction of missense below the ESM1b disruptive threshold.",
    "esm_n_annotated": "Number of missense variants with an ESM1b score.",
    "af_n_syn": "Count of synonymous variants with allele frequency.",
    "af_median_log10_syn": "Median log10 AF of synonymous variants.",
    "af_frac_singleton_syn": "Singleton fraction among synonymous variants.",
    "af_frac_common_syn": "Common-variant fraction among synonymous variants.",
    "af_median_log10_mis": "Median log10 AF of missense variants.",
    "af_frac_singleton_mis": "Singleton fraction among missense variants.",
    "af_frac_common_mis": "Common-variant fraction among missense variants.",
    "af_log2ratio_mis_syn_count": "Log2 missense:synonymous count ratio.",
    "af_log2ratio_lof_syn_count": "Log2 LoF:synonymous count ratio.",
    "af_median_log10_delta_mis_syn": "Missense-minus-synonymous shift in median log10 AF.",
    "af_frac_rare_delta_mis_syn": "Missense-minus-synonymous shift in rare-variant fraction.",
    "af_rg_n_variants": "Variants falling within RG motifs (with AF).",
    "af_rg_median_log10": "Median log10 AF of RG-motif variants.",
    "af_rg_frac_singleton": "Singleton fraction among RG-motif variants.",
    "af_rg_vs_nonrg_log10_delta_mis": "RG-vs-non-RG shift in median log10 missense AF.",
    "af_am_score_weighted_by_rarity": "AlphaMissense score weighted by variant rarity.",
    "af_n_likely_path_rare": "Count of rare likely-pathogenic variants.",
    "rg_event_fraction_no_change": "Fraction of RG events leaving the motif unchanged.",
    "rg_event_fraction_gain": "Fraction of RG events creating a motif.",
    "rg_event_fraction_movement": "Fraction of RG events shifting a motif.",
    "delta_rg_ratio_rel_mean": "Mean relative change in RG ratio across variants.",
    "rg_fraction_rgs_hit_LoF": "Fraction of RG sites hit by an LoF variant.",
    "rg_fraction_rgs_hit_inframe_indel": "Fraction of RG sites hit by an in-frame indel.",
    "rg_fraction_rgs_hit_missense": "Fraction of RG sites hit by a missense variant.",
    "rg_fraction_rgs_hit_synonymous": "Fraction of RG sites hit by a synonymous variant.",
    "rg_mean_burden_on_hit_LoF": "Mean LoF burden on hit RG sites.",
    "rg_mean_burden_on_hit_inframe_indel": "Mean in-frame indel burden on hit RG sites.",
    "rg_mean_burden_on_hit_missense": "Mean missense burden on hit RG sites.",
    "rg_mean_burden_on_hit_synonymous": "Mean synonymous burden on hit RG sites.",
    "n_g_hits_disrupting": "Count of disrupting variants at glycine positions.",
    "n_r_hits_disrupting": "Count of disrupting variants at arginine positions.",
    "rg_r_fraction": "Fraction of RG-motif residues that are arginine.",
    "delta_ncpr": "Mean variant-induced shift in net charge per residue.",
    "delta_fcr": "Mean shift in fraction of charged residues.",
    "delta_kappa": "Mean shift in charge patterning.",
    "delta_hydropathy": "Mean shift in hydropathy.",
    "delta_aromaticity": "Mean shift in aromaticity.",
    "delta_fraction_proline": "Mean shift in proline fraction.",
    "delta_n_pos": "Mean shift in positive-residue count.",
    "delta_n_neg": "Mean shift in negative-residue count.",
}
# add codon_* descriptions programmatically
for c in all_feats:
    if c.startswith("codon_") and c not in DESCRIPTIONS:
        parts = c.split("_")  # codon_<AA>_<CODON>
        if len(parts) == 3:
            DESCRIPTIONS[c] = f"Usage fraction of codon {parts[2]} for {parts[1]} within the region."

# ── 3) build the retained-feature matrix exactly as the model sees it ───────
X_ret = X_static[feat_cols].copy()
X_imp = imp.transform(X_ret.values)
y_arr = y.reindex(X_static.index).values

# ── 4a) SHAP global importance (mean |SHAP|) on retained features ───────────
explainer = shap.TreeExplainer(rf)
sv = explainer.shap_values(X_imp)
if isinstance(sv, list):
    sv_pos = sv[1]
elif np.ndim(sv) == 3:
    sv_pos = sv[:, :, 1]
else:
    sv_pos = sv
mean_abs_shap = np.abs(sv_pos).mean(axis=0)   # one value per retained feature
shap_imp = dict(zip(feat_cols, mean_abs_shap))

# ── 4b) permutation importance (AUC drop) on retained features ──────────────
auc_scorer = make_scorer(roc_auc_score, needs_proba=True)
perm = permutation_importance(
    rf, X_imp, y_arr,
    scoring=auc_scorer, n_repeats=20, random_state=42, n_jobs=-1,
)
perm_imp     = dict(zip(feat_cols, perm.importances_mean))
perm_imp_std = dict(zip(feat_cols, perm.importances_std))

# ── 5) assemble the table ────────────────────────────────────────────────────
rows = []
for f in all_feats:
    dropped = f in pruned
    rows.append({
        "feature": f,
        "group": feat_to_group.get(f, "ungrouped"),
        "description": DESCRIPTIONS.get(f, ""),
        "dropped_by_pruning": dropped,
        "mean_abs_shap": np.nan if dropped else shap_imp.get(f, np.nan),
        "perm_importance_auc": np.nan if dropped else perm_imp.get(f, np.nan),
        "perm_importance_std": np.nan if dropped else perm_imp_std.get(f, np.nan),
    })

tbl = pd.DataFrame(rows)
# sort: retained first, by SHAP desc; pruned last
tbl = tbl.sort_values(
    ["dropped_by_pruning", "mean_abs_shap"],
    ascending=[True, False],
).reset_index(drop=True)

# flag any feature missing a description so you don't ship blanks
missing_desc = tbl.loc[tbl["description"] == "", "feature"].tolist()
if missing_desc:
    print(f"[!] {len(missing_desc)} features need descriptions: {missing_desc}")

OUT = "/mnt/d/phd/scripts/16_ev_signature_predictor/data/output/feature_table_supplementary.csv"
tbl.to_csv(OUT, sep=",", index=False)
print(f"wrote {len(tbl)} features ({(~tbl.dropped_by_pruning).sum()} retained, "
      f"{tbl.dropped_by_pruning.sum()} pruned) -> {OUT}")
print(tbl.head(20).to_string(index=False))